In [92]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter
import os

In [93]:
os.chdir(r"C:\Users\hempe\Studium\Masterthesis\Repository\Masterthesis")
# BindingDB_BindingDB_Articles.tsv mit UTF-8-Encoding laden
df = pd.read_csv('data/raw/BindingDB_Patents.tsv', sep='\t', encoding='utf-8',low_memory=False)

In [94]:
columns=df.columns.tolist()

In [95]:
#Select relevant feaures
df=df[['Ligand SMILES', 'BindingDB Target Chain Sequence 1', 'IC50 (nM)', 'Ki (nM)', 'Kd (nM)', 'BindingDB MonomerID','BindingDB Reactant_set_id','Target Name']]
df.head()

,Ligand SMILES,BindingDB Target Chain Sequence 1,IC50 (nM),Ki (nM),Kd (nM),BindingDB MonomerID,BindingDB Reactant_set_id,Target Name
0,Cc1c(Br)c(=O)oc2c(C=O)c(O)ccc12,MPARRLLLLLTLLLPGLGIFGSTSTVTLPETLLFVSTLDGSLHAVS...,<100,NaN,NaN,160334,455290,Serine/threonine-protein kinase/endoribonuclea...
1,CN1CCN(CC1)C(=O)c1ccc(cc1)-c1c(C)c2ccc(O)c(C=O...,MPARRLLLLLTLLLPGLGIFGSTSTVTLPETLLFVSTLDGSLHAVS...,<100,NaN,NaN,160335,455291,Serine/threonine-protein kinase/endoribonuclea...
2,Cc1c(-c2ccc(cc2)C(=O)NCCN2CCOCC2)c(=O)oc2c(C=O...,MPARRLLLLLTLLLPGLGIFGSTSTVTLPETLLFVSTLDGSLHAVS...,<100,NaN,NaN,51439,455292,Serine/threonine-protein kinase/endoribonuclea...
3,CCN(CC)CCc1c(C)c2ccc(O)c(C=O)c2oc1=O,MPARRLLLLLTLLLPGLGIFGSTSTVTLPETLLFVSTLDGSLHAVS...,<100,NaN,NaN,160337,455293,Serine/threonine-protein kinase/endoribonuclea...
4,CC(C)c1cc(=O)oc2c(C=O)c(O)ccc12,MPARRLLLLLTLLLPGLGIFGSTSTVTLPETLLFVSTLDGSLHAVS...,<100,NaN,NaN,160338,455294,Serine/threonine-protein kinase/endoribonuclea...


In [96]:
# zahle alle spalten in denen Kd (nM) nicht null sind
non_null_kd = df['Kd (nM)'].notnull().sum()
non_null_ki = df['Ki (nM)'].notnull().sum()
non_null_ic50 = df['IC50 (nM)'].notnull().sum() 

df_counts = pd.DataFrame({
    "Messgröße": ["Kd (nM)", "Ki (nM)", "IC50 (nM)"],
    "Anzahl nicht-null": [non_null_kd, non_null_ki, non_null_ic50]
})

df_counts

,Messgröße,Anzahl nicht-null
0,Kd (nM),32905
1,Ki (nM),160932
2,IC50 (nM),1061229


In [97]:
# Alle Datensätze mit fehlenden Werten in den Spalten
# IC50 (nM), BindingDB Target Chain Sequence 1 und Ligand SMILES entfernen
df = df[[ 'Ligand SMILES', 'BindingDB Target Chain Sequence 1', 'Kd (nM)']].dropna()

In [98]:
df.shape

(32905, 3)

In [99]:
anzahl_duplikate = df.duplicated(
    subset=["Ligand SMILES", "BindingDB Target Chain Sequence 1", "Kd (nM)"]
).sum()

In [100]:
anzahl_duplikate

3491

In [108]:

# zähle spalten mit eindeutigen Werten in der Spalte 'BindingDB Target Chain Sequence 1'
unique_counts_protein = df['BindingDB Target Chain Sequence 1'].nunique()
unique_counts_protein

# zähle spalten mit eindeutigen Werten in der Spalte 'Ligand SMILES'
# zähle spalten mit eindeutigen Werten in der Spalte 'Ligand SMILES'
unique_counts_ligand = df['Ligand SMILES'].nunique()
unique_counts_ligand

df_duplicates = pd.DataFrame({
    "Messgröße": ["Proteins", "Ligands", "Reactant_set"],
    "Number of values": [unique_counts_protein, unique_counts_ligand, anzahl_duplikate]
})

df_duplicates

,Messgröße,Number of values
0,Proteins,332
1,Ligands,14488
2,Reactant_set,3491


In [102]:
df['Kd (nM)'].describe()

count     32905
unique     3241
top        <0.2
freq       3086
Name: Kd (nM), dtype: object

In [103]:
# Originalwerte sichern
ic50_original = df['Kd (nM)']

# Umwandlung testen
ic50_numeric = pd.to_numeric(ic50_original, errors="coerce")

# Zeilen finden, bei denen die Umwandlung fehlgeschlagen ist
non_numeric_rows = df[ic50_numeric.isna() & ic50_original.notna()]

# Einzigartige nicht-numerische Werte anzeigen
non_numeric_rows['Kd (nM)'].value_counts().head(50)

Kd (nM)
<0.2          3086
<10            805
>30000         599
>10000         558
>10            460
<250           448
<200           422
>100000        410
<1.000         402
<1             398
<5             382
>2500          329
>1000          311
<10000         287
<50            225
<500           207
<100000        189
<100           174
>100           158
>300           156
<0.1           150
<1000           94
<300            92
<50000          90
>800            72
>100000000      68
>30200          62
>5000           50
<30000          50
<20000          43
>1000000        43
>500            38
<0.3            37
<30             31
>20000          31
>200            30
<30.0           28
>3020           28
<500000         20
>50000          19
>1.0            17
>3000           14
>30000000       13
>200000         12
>150000         11
>500000         11
<25.0           11
>50             11
<10.00          10
>404000          9
Name: count, dtype: int64

In [104]:
# Spalte IC50 (nM) in numerische Werte umwandeln
# Nicht-numerische Werte werden zu NaN und anschließend entfernt
df['Kd (nM)'] = pd.to_numeric(df['Kd (nM)'], errors='coerce')
df = df.dropna(subset=['Kd (nM)'])

In [107]:

# zähle spalten mit eindeutigen Werten in der Spalte 'BindingDB Target Chain Sequence 1'
unique_counts_protein = df['BindingDB Target Chain Sequence 1'].nunique()
unique_counts_protein

# zähle spalten mit eindeutigen Werten in der Spalte 'Ligand SMILES'
# zähle spalten mit eindeutigen Werten in der Spalte 'Ligand SMILES'
unique_counts_ligand = df['Ligand SMILES'].nunique()
unique_counts_ligand

df_duplicates = pd.DataFrame({
    "Messgröße": ["Proteins", "Ligands", "Reactant_set"],
    "Anzahl nicht-null": [unique_counts_protein, unique_counts_ligand, anzahl_duplikate]
})

df_duplicates

,Messgröße,Anzahl nicht-null
0,Proteins,332
1,Ligands,14488
2,Reactant_set,3491


In [105]:
df.shape

(21651, 3)

In [106]:
df['Kd (nM)'].describe()

count    2.165100e+04
mean     9.492590e+04
std      3.371812e+06
min      0.000000e+00
25%      2.000000e+00
50%      1.700000e+01
75%      4.420000e+02
max      3.520000e+08
Name: Kd (nM), dtype: float64